# Phase 01 — ETL + đồ thị CSE-CIC-IDS2018-v2 trên Colab

Chạy sau `00_colab_bootstrap.ipynb` (mount Drive, clone repo, cài thư viện). Notebook này tạo artifact **protocol mới** (`processed_v2`): split chung theo khối, `flow_id`, scaler chỉ fit trên train, nhãn binary/multiclass. Không ghi đè `data/processed` của thực tập.

**Yêu cầu:** raw CSV đã có ở `DRIVE_ROOT/data/raw/nf-cse-cic-ids2018-v2/` (xem bootstrap). Nên dùng runtime **High-RAM** nếu Colab báo thiếu RAM (ETL ước tính cần khoảng 10 GB; chưa đo trên toàn bộ dữ liệu).

Cấu hình đầu ra: ghi vào đĩa cục bộ `/content/processed_v2` (nhanh), rồi copy lên Drive ở cuối để không mất khi phiên bị ngắt.

In [ ]:
import os, sys
GITHUB_USER, GITHUB_REPO = "thunww", "gnn-nids-tttn"
DRIVE_ROOT = "/content/drive/MyDrive/GNN-NIDS_TTTN"
PROJECT_DIR = "/content/GNN-NIDS_TTTN"
DATASET = "nf-cse-cic-ids2018-v2"
LOCAL_OUT = "/content/processed_v2"
DRIVE_OUT = f"{DRIVE_ROOT}/data/processed_v2"

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
if os.path.isdir(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull
else:
    !git clone https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git {PROJECT_DIR}
!pip install -q -r {PROJECT_DIR}/requirements.txt
sys.path.insert(0, f"{PROJECT_DIR}/src")

## 1. Kiểm tra nhanh trước khi chạy dài (test + mẫu nhỏ)

In [ ]:
!cd {PROJECT_DIR} && PYTHONPATH=src python -m pytest src -q -p no:cacheprovider 2>&1 | tail -3
!cd {PROJECT_DIR} && PYTHONPATH=src python src/etl/run_etl_v2.py --dataset {DATASET} --raw-dir {DRIVE_ROOT}/data/raw --max-rows 300000 --out /content/sample_v2 2>&1 | tail -15

## 2. ETL đầy đủ

Đọc CSV 3,2 GB, loại trùng/thiếu, gán `flow_id`, chia khối, fit scaler trên train, ghi `flows.parquet`, `raw_test.parquet`, `contract.json`, `audit_etl.json`, `class_support_*.csv`, `novel_flow_ids.parquet`.

In [ ]:
!cd {PROJECT_DIR} && PYTHONPATH=src python src/etl/run_etl_v2.py --dataset {DATASET} --raw-dir {DRIVE_ROOT}/data/raw --out {LOCAL_OUT}

### Xem kết quả audit — **bắt buộc đọc trước khi build đồ thị**

Chú ý `classes_missing_by_split`: lớp nào bằng 0 ở train/val/test (tấn công CSE-CIC tập trung theo ngày nên chia khối có thể làm lớp vắng). Và `cross_split_overlap` (tỷ lệ flow val/test trùng vector đặc trưng với train).

In [ ]:
import json, pandas as pd
audit = json.load(open(f"{LOCAL_OUT}/{DATASET}/audit_etl.json", encoding="utf-8"))
print(json.dumps(audit, ensure_ascii=False, indent=2))
pd.read_csv(f"{LOCAL_OUT}/{DATASET}/class_support_multiclass.csv")

## 3. Dựng đồ thị (binary và multiclass)

Ghi shard tăng dần (256 đồ thị/shard). `--workers` nên ≤ số CPU của runtime; giảm xuống 2 nếu thiếu RAM.

In [ ]:
for task in ("binary", "multiclass"):
    !cd {PROJECT_DIR} && PYTHONPATH=src python src/graph/run_graph_builder_v2.py --dataset {DATASET} --task {task} --processed {LOCAL_OUT} --workers 2

## 4. Kiểm chứng artifact

Nạp lại shard: nhãn `y` khớp nhãn gốc tra theo `flow_id`; số `flow_id` giao nhau giữa các split phải bằng 0.

In [ ]:
import glob, torch, numpy as np, pandas as pd
from etl.protocol import assert_disjoint_splits

task = "multiclass"
truth = pd.read_parquet(f"{LOCAL_OUT}/{DATASET}/flows.parquet", columns=["flow_id", "target_" + task]).set_index("flow_id")["target_" + task]
ids = {}
for s in ("train", "val", "test"):
    fl, ok = [], True
    for f in sorted(glob.glob(f"{LOCAL_OUT}/{DATASET}/graphs/{task}/{s}_shard_*.pt")):
        for g in torch.load(f, weights_only=False):
            fl.append(g.flow_id.numpy())
            ok &= bool((truth.loc[g.flow_id.numpy()].to_numpy() == g.y.numpy()).all())
    ids[s] = np.unique(np.concatenate(fl))
    print(s, "flows trong graph:", len(ids[s]), "| nhan khop:", ok)
assert_disjoint_splits(ids)
print("flow_id giao nhau giua cac split: 0 -> OK")

## 5. Baseline RF/XGBoost trên split chung (đánh giá VAL, chưa chạm test)

Chạy thử với **1 triệu flow train** (ghi rõ trong `baseline_metrics_val.json`) vì RF trên 13 triệu dòng rất tốn RAM/thời gian. Bản đầy đủ chạy sau nếu cần.

In [ ]:
for task in ("binary", "multiclass"):
    !cd {PROJECT_DIR} && PYTHONPATH=src python src/models/train_baseline_v2.py --dataset {DATASET} --task {task} --processed {LOCAL_OUT}

## 6. Lưu lên Drive

Copy toàn bộ sang Drive (an toàn khi phiên Colab bị ngắt). Các lần sau bootstrap có thể copy ngược về `/content` để train.

In [ ]:
import shutil
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copytree(f"{LOCAL_OUT}/{DATASET}", f"{DRIVE_OUT}/{DATASET}", dirs_exist_ok=True)
print("Da luu:", os.listdir(f"{DRIVE_OUT}/{DATASET}"))